<figure>
<center>
<img src='https://www.economicas.uba.ar/wp-content/uploads/2020/08/cropped-logo_FCE.png' />
</figure>

# Validar los parámetros de funciones y métodos
**Taller de Programación para el Análisis de Datos — FCE-UBA**

Cuando escribimos una función, esperamos que le pasen ciertos datos: un número, un texto, una lista... Pero Python **no controla** qué le pasan. En esta notebook vemos cómo hacer ese control, de dos formas:

1. **A mano**, con `isinstance` y `raise`.
2. **Automático**, con el decorador `@validate_call` de la librería `pydantic`.

In [ ]:
# Instalación (descomentar si hace falta, por ejemplo en Colab)
# %pip install pydantic

---
## 1. El problema

Una función simple que calcula el total de una venta:

In [ ]:
def calcular_total(precio, cantidad):
    return precio * cantidad

print(calcular_total(1500, 2))     # lo esperado
print(calcular_total("1500", 2))   # ¡no da error!

3000
15001500


El segundo caso no da error: Python repite el texto `"1500"` dos veces y devuelve `"15001500"`. El resultado es incorrecto y nadie se entera.

> **Validar** significa revisar los datos al entrar a la función y, si están mal, **cortar con un error claro**.

---
## 2. Validar a mano con `isinstance`

`isinstance(valor, tipo)` devuelve `True` si el valor es de ese tipo:

In [ ]:
print(isinstance(1500, int))      # True
print(isinstance("1500", int))    # False
print(isinstance(1500.5, float))  # True

# Se puede preguntar por varios tipos a la vez, usando una tupla
print(isinstance(1500, (int, float)))    # True

True
False
True
True


Lo usamos al principio de la función. Si el dato está mal, lanzamos un error con `raise`:

In [ ]:
def calcular_total(precio, cantidad):
    if not isinstance(precio, (int, float)):
        raise TypeError("precio tiene que ser un número")
    if not isinstance(cantidad, int):
        raise TypeError("cantidad tiene que ser un entero")

    return precio * cantidad

print(calcular_total(1500, 2))

3000


In [ ]:
try:
    calcular_total("1500", 2)
except TypeError as error:
    print("ERROR:", error)

ERROR: precio tiene que ser un número


Ahora el error aparece enseguida y dice exactamente qué pasó.

### 2.1 Validar también los valores

Además del **tipo**, a veces importa el **valor**: un precio no puede ser negativo. Para eso usamos `ValueError`:

| Error | Cuándo usarlo | Ejemplo |
|---|---|---|
| `TypeError` | El dato es del **tipo** equivocado | `"1500"` en lugar de `1500` |
| `ValueError` | El tipo es correcto, pero el **valor** no sirve | `-1500` como precio |

In [ ]:
def calcular_total(precio, cantidad):
    if not isinstance(precio, (int, float)):
        raise TypeError("precio tiene que ser un número")
    if not isinstance(cantidad, int):
        raise TypeError("cantidad tiene que ser un entero")
    if precio <= 0:
        raise ValueError("precio tiene que ser mayor a 0")
    if cantidad < 1:
        raise ValueError("cantidad tiene que ser al menos 1")

    return precio * cantidad

### 2.2 Capturar el error con `try` / `except`

Para que el programa no se detenga, podemos capturar el error y mostrar un mensaje:

In [ ]:
pruebas = [(1500, 2), ("1500", 2), (1500, 2.5), (-100, 2), (1500, 0)]

for precio, cantidad in pruebas:
    try:
        total = calcular_total(precio, cantidad)
        print(f"{precio!r}, {cantidad!r} -> total = {total}")
    except (TypeError, ValueError) as error:
        print(f"{precio!r}, {cantidad!r} -> ERROR: {error}")

1500, 2 -> total = 3000
'1500', 2 -> ERROR: precio tiene que ser un número
1500, 2.5 -> ERROR: cantidad tiene que ser un entero
-100, 2 -> ERROR: precio tiene que ser mayor a 0
1500, 0 -> ERROR: cantidad tiene que ser al menos 1


**Ventaja:** no hace falta instalar nada.
**Desventaja:** por cada parámetro hay que escribir varios `if`. Con funciones de muchos parámetros se vuelve largo y repetitivo.

---
## 3. Anotaciones de tipo (*type hints*)

Python permite **escribir** en la definición de la función qué tipo esperamos en cada parámetro:

In [ ]:
def calcular_total(precio: float, cantidad: int) -> float:
    return precio * cantidad

Se lee así:
- `precio: float` → `precio` debería ser un número decimal.
- `cantidad: int` → `cantidad` debería ser un entero.
- `-> float` → la función devuelve un número decimal.

**Pero ojo: Python no las controla.** Son solo una indicación, como un comentario:

In [ ]:
print(calcular_total("1500", 2))   # sigue sin dar error

15001500


Entonces, ¿para qué sirven? Para que quien lea el código sepa qué pasarle, para que el editor (VS Code, Colab) nos ayude con sugerencias... y para que otras herramientas **sí las controlen**. Eso es lo que vemos a continuación.

---
## 4. Validar automáticamente con `@validate_call`

La librería `pydantic` tiene un **decorador** llamado `@validate_call`. Un decorador es una línea con `@` que se pone arriba de la función y le agrega un comportamiento.

En este caso, `@validate_call` **lee las anotaciones de tipo y las controla** cada vez que se llama a la función:

In [ ]:
from pydantic import validate_call, ValidationError

@validate_call
def calcular_total(precio: float, cantidad: int) -> float:
    return precio * cantidad

print(calcular_total(1500, 2))

3000.0


In [ ]:
try:
    calcular_total(precio="mil", cantidad=2)
except ValidationError as error:
    print(error)

1 validation error for calcular_total
precio
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='mil', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/float_parsing


Con **una sola línea** (`@validate_call`) obtuvimos el control de tipos, sin escribir ningún `if`. El error dice qué parámetro falló (`precio`), por qué y qué valor llegó (`input_value='mil'`).

> Si llamamos a la función sin los nombres, `calcular_total("mil", 2)`, el error muestra la **posición** del parámetro en lugar del nombre: `0` es el primero, `1` el segundo, etc.

### 4.1 Convierte cuando tiene sentido

Algo práctico de pydantic: si recibe `"1500"` (un texto que **representa** un número), lo convierte a número. Esto es muy útil con datos que vienen de archivos CSV o formularios, donde todo llega como texto.

In [ ]:
print(calcular_total("1500", "2"))   # convierte "1500" -> 1500.0 y "2" -> 2

3000.0


In [ ]:
pruebas = [(1500, 2), ("1500", 2), ("mil", 2), (1500, 2.5), (1500, "dos")]

for precio, cantidad in pruebas:
    try:
        total = calcular_total(precio, cantidad)
        print(f"{precio!r}, {cantidad!r} -> total = {total}")
    except ValidationError as error:
        print(f"{precio!r}, {cantidad!r} -> ERROR")

1500, 2 -> total = 3000.0
'1500', 2 -> total = 3000.0
'mil', 2 -> ERROR
1500, 2.5 -> ERROR
1500, 'dos' -> ERROR


Noten que `2.5` como cantidad da error: no se puede convertir a entero sin perder información.

### 4.2 Validar valores: mayor a 0, entre 1 y 100...

Para controlar también el **valor**, usamos `Field` dentro de `Annotated`. Se lee así:

`Annotated[float, Field(gt=0)]` → "un `float` que además tiene que ser mayor a 0"

| En `Field` | Significa |
|---|---|
| `gt=0` | mayor a 0 (*greater than*) |
| `ge=1` | mayor o igual a 1 (*greater or equal*) |
| `lt=100` | menor a 100 (*less than*) |
| `le=100` | menor o igual a 100 (*less or equal*) |
| `min_length=2` | texto de al menos 2 caracteres |

In [ ]:
from typing import Annotated
from pydantic import Field

@validate_call
def calcular_total(
    precio: Annotated[float, Field(gt=0)],
    cantidad: Annotated[int, Field(ge=1, le=100)],
) -> float:
    return precio * cantidad

print(calcular_total(1500, 2))

3000.0


In [ ]:
try:
    calcular_total(precio=-100, cantidad=500)
except ValidationError as error:
    print(error)

2 validation errors for calcular_total
precio
  Input should be greater than 0 [type=greater_than, input_value=-100, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than
cantidad
  Input should be less than or equal to 100 [type=less_than_equal, input_value=500, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


Detectó **los dos errores a la vez**, cosa que con los `if` manuales no pasaba (se cortaba en el primero).

### 4.3 Parámetros opcionales y listas

Los valores por defecto funcionan igual que siempre. También se pueden validar listas: `list[float]` significa "una lista de números".

In [ ]:
@validate_call
def aplicar_descuento(total: float, descuento: Annotated[float, Field(ge=0, le=0.5)] = 0.0) -> float:
    return total * (1 - descuento)

print(aplicar_descuento(1000))          # usa descuento = 0.0
print(aplicar_descuento(1000, 0.1))

1000.0
900.0


In [ ]:
@validate_call
def promedio(valores: list[float]) -> float:
    return sum(valores) / len(valores)

print(promedio([10, 20, "30"]))   # "30" se convierte a 30.0

try:
    promedio(valores=[10, 20, "treinta"])
except ValidationError as error:
    print(error)

20.0
1 validation error for promedio
valores.2
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='treinta', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/float_parsing


`valores.2` significa: en el parámetro `valores`, el elemento de la posición 2.

---
## 5. Validar en los métodos de una clase

Todo lo anterior también sirve para los **métodos** de una clase, incluido `__init__` (el que se ejecuta al crear el objeto).

### 5.1 A mano

In [ ]:
class Producto:
    def __init__(self, nombre, precio):
        if not isinstance(nombre, str):
            raise TypeError("nombre tiene que ser un texto")
        if not isinstance(precio, (int, float)):
            raise TypeError("precio tiene que ser un número")
        if precio <= 0:
            raise ValueError("precio tiene que ser mayor a 0")
        self.nombre = nombre
        self.precio = precio

    def total(self, cantidad):
        if not isinstance(cantidad, int) or cantidad < 1:
            raise ValueError("cantidad tiene que ser un entero mayor o igual a 1")
        return self.precio * cantidad

yerba = Producto("yerba", 1500)
print(yerba.total(3))

4500


In [ ]:
try:
    Producto("café", -10)
except (TypeError, ValueError) as error:
    print("ERROR:", error)

ERROR: precio tiene que ser mayor a 0


### 5.2 Con `@validate_call`

Se pone el decorador arriba de cada método. El parámetro `self` no se valida, solo los demás:

In [ ]:
class Producto:
    @validate_call
    def __init__(self, nombre: Annotated[str, Field(min_length=2)], precio: Annotated[float, Field(gt=0)]):
        self.nombre = nombre
        self.precio = precio

    @validate_call
    def total(self, cantidad: Annotated[int, Field(ge=1)]) -> float:
        return self.precio * cantidad

yerba = Producto("yerba", "1500")    # "1500" se convierte a 1500.0
print(yerba.precio)
print(yerba.total(3))

1500.0
4500.0


In [ ]:
pruebas = [("café", -10), ("x", 900), (123, 900)]

for nombre, precio in pruebas:
    try:
        Producto(nombre=nombre, precio=precio)
        print(f"{nombre!r}, {precio!r} -> OK")
    except ValidationError as error:
        print(f"{nombre!r}, {precio!r} -> ERROR")
        print(error, "\n")

'café', -10 -> ERROR
1 validation error for Producto.__init__
precio
  Input should be greater than 0 [type=greater_than, input_value=-10, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than 

'x', 900 -> ERROR
1 validation error for Producto.__init__
nombre
  String should have at least 2 characters [type=string_too_short, input_value='x', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short 

123, 900 -> ERROR
1 validation error for Producto.__init__
nombre
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type 



> Noten que `123` como nombre **no** se convierte a `"123"`: pydantic no transforma números en texto automáticamente.

---
## 6. Ejercicios

1. Escribí una función `calcular_iva(monto, tasa)` que valide **a mano** que `monto` sea un número mayor o igual a 0 y que `tasa` sea `0.105` o `0.21`. Probala con valores correctos e incorrectos usando `try` / `except`.

2. Reescribí la función del ejercicio 1 usando `@validate_call` y `Field`. (Pista: para que `tasa` solo acepte esos dos valores podés usar `Literal[0.105, 0.21]`, importando `Literal` desde `typing`.)

3. Escribí una función `nota_final(parcial1, parcial2)` con `@validate_call` donde cada nota sea un número entre 1 y 10. Debe devolver el promedio. ¿Qué pasa si le pasás `"8"`? ¿Y `"ocho"`?

4. Creá una clase `Alumno` con `nombre` (texto de al menos 2 letras) y `edad` (entero entre 17 y 99), validados en `__init__` con `@validate_call`. Agregale un método `es_mayor_de(edad_minima)` que también valide su parámetro.